In [4]:
import h5netcdf

fn = "./13901616_nc/W_12hmean_coarse.nc"

with h5netcdf.File(fn, "r") as nc:
    print(nc)
    tlen = nc.variables["time"].shape[0]
    vlen = nc.variables["W"].shape[0]
    print("time length:", nc.variables["time"][-1])
    print("data length:", vlen)
    print("equal?", tlen == vlen)

<h5netcdf.File 'W_12hmean_coarse.nc' (mode r)>
Dimensions:
    k_l: <h5netcdf.Dimension 'k_l': size 49>
    j: <h5netcdf.Dimension 'j': size 120>
    i: <h5netcdf.Dimension 'i': size 60>
    time: <h5netcdf.Dimension 'time': size 7179 (unlimited)>
Groups:
Variables:
    W: ('time', 'k_l', 'j', 'i') float32
    time: ('time',) int64
Attributes:
    origin_iter: 13901616
    hourly_step: 144
    block_size: 12
    time_is_block_start: 1
    box_j: 10
    box_i: 10
    varname: 'W'
    split_note: 'time >= 13900000'
time length: 26305200
data length: 7179
equal? True


In [1]:
import os
import h5netcdf
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

import xarray as xr
import numpy as np

from src.utils import make_grid
from src.utils import build_coarse_grid
from src.utils import read_mitgcm
from src.utils.coarsen_UVWTS import coarse_field
from MITgcmutils import mds
from src.utils import mld_avg
from MITgcmutils import mds


def make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var: str, dims):
    """
    Returns load_snapshot(it) -> xr.DataArray (3D) for ONE variable.
    This is faster than loading a full Dataset each time.
    """
    def load_snapshot(it: int) -> xr.DataArray:
        da = read_mitgcm._ocean_var(grid_dir, var, it, dims)
        da = da.drop_vars("time", errors="ignore")
        da.name = var
        return da
    return load_snapshot


def _12h_mean_to_netcdf_onevar(
    start_time_index: int,
    end_time_index: int,
    load_snapshot,                   # fn(it:int) -> xr.DataArray (3D)
    out_nc: str,                     # e.g. "./out_nc/T_12hmean_coarse.nc"
    ds_fine: xr.Dataset,
    box_j: int,
    box_i: int,
    varname: str,                    # e.g. "T"
    block_size: int = 12,
    step: int = 144,
    time_name: str = "time",
    dtype=np.float32,
    skip_missing: bool = False,
    print_every: int = 20,
):
    start_time_index = int(start_time_index)
    end_time_index   = int(end_time_index)
    step             = int(step)

    if end_time_index < start_time_index:
        raise ValueError("end_time_index must be >= start_time_index")

    out_dir = os.path.dirname(out_nc) or "."
    os.makedirs(out_dir, exist_ok=True)

    block_span = block_size * step

    # ---- read existing file info (saved block starts + origin alignment)
    saved = set()
    origin_iter = start_time_index

    if os.path.exists(out_nc):
        with h5netcdf.File(out_nc, "r") as nc:
            if time_name in nc.variables:
                saved = set(map(int, nc.variables[time_name][:].tolist()))
            if "origin_iter" in nc.attrs:
                origin_iter = int(nc.attrs["origin_iter"])

            # optional safety: ensure the file matches this variable/coarsen setup
            if "varname" in nc.attrs and str(nc.attrs["varname"]) != str(varname):
                raise ValueError(f"{out_nc} already stores varname={nc.attrs['varname']} but you requested {varname}")
            if "box_j" in nc.attrs and int(nc.attrs["box_j"]) != int(box_j):
                raise ValueError("box_j mismatch with existing file")
            if "box_i" in nc.attrs and int(nc.attrs["box_i"]) != int(box_i):
                raise ValueError("box_i mismatch with existing file")

    # last hour of block must be <= end_time_index
    max_n = int(np.floor((end_time_index - origin_iter - (block_size - 1) * step) / block_span))
    if max_n < 0:
        print("Not enough hourly data to form even one full 12-hour block yet.")
        return

    # ---- init file (for ONE variable only)
    def _init_file(coarse_da: xr.DataArray):
        # coarse_da is 3D (no time dim)
        spatial_dims = tuple(coarse_da.dims)
        spatial_sizes = {d: int(coarse_da.sizes[d]) for d in spatial_dims}

        with h5netcdf.File(out_nc, "w") as nc:
            # dimensions
            nc.dimensions[time_name] = None
            for d, n in spatial_sizes.items():
                nc.dimensions[d] = n

            # attrs
            nc.attrs["origin_iter"] = int(origin_iter)
            nc.attrs["hourly_step"] = int(step)
            nc.attrs["block_size"] = int(block_size)
            nc.attrs["time_is_block_start"] = 1
            nc.attrs["box_j"] = int(box_j)
            nc.attrs["box_i"] = int(box_i)
            nc.attrs["varname"] = str(varname)

            # time coord
            tvar = nc.create_variable(
                time_name,
                dimensions=(time_name,),
                dtype="i8",
                chunks=(1,),
                maxshape=(None,),
            )
            tvar.attrs["long_name"] = "start index of each 12-hour averaging block"
            tvar.attrs["units"] = "model_iteration_index"

            # data var: (time, spatial...)
            dims = (time_name,) + spatial_dims
            chunks = (1,) + tuple(spatial_sizes[d] for d in spatial_dims)
            maxshape = (None,) + tuple(spatial_sizes[d] for d in spatial_dims)

            v = nc.create_variable(
                varname,
                dimensions=dims,
                dtype="f4",
                chunks=chunks,
                maxshape=maxshape,
            )
            v.attrs["long_name"] = varname

    appended = 0
    skipped_saved = 0
    skipped_missing = 0

    nc_app = None
    try:
        for n in range(max_n + 1):
            s = origin_iter + n * block_span

            if print_every and (n % print_every == 0):
                print(f"[try] n={n}/{max_n}  block_start={s}", flush=True)

            if s in saved:
                skipped_saved += 1
                continue

            block_iters = [s + m * step for m in range(block_size)]

            # ---- fast 12h mean for ONE var (sum numpy arrays)
            snap0 = None
            sum_arr = None

            try:
                for it in block_iters:
                    da = load_snapshot(int(it))  # xr.DataArray (3D)

                    if snap0 is None:
                        snap0 = da
                        sum_arr = da.values.astype(dtype, copy=True)
                    else:
                        sum_arr += da.values.astype(dtype, copy=False)

            except Exception as e:
                if skip_missing:
                    skipped_missing += 1
                    print(f"  -> missing/unreadable snapshot in block_start={s}, skip ({e})", flush=True)
                    continue
                raise

            mean_arr = sum_arr / block_size

            # rebuild mean DataArray with same coords/dims as first snapshot
            mean_da = snap0.copy(data=mean_arr)
            mean_da.name = varname

            # ---- coarsen AFTER averaging
            coarse_da = coarse_field(varname, mean_da, ds_fine, box_j=box_j, box_i=box_i)
            coarse_da = coarse_da.astype("float32")
            coarse_da.name = varname

            # ---- initialize file once we know coarse shape
            if not os.path.exists(out_nc):
                _init_file(coarse_da)

            # ---- open once for appending
            if nc_app is None:
                nc_app = h5netcdf.File(out_nc, "a")

            # append index
            tvar = nc_app.variables[time_name]
            write_idx = int(tvar.shape[0])

            # resize variable first, write data
            v = nc_app.variables[varname]
            spatial_shape = v.shape[1:]
            v._h5ds.resize((write_idx + 1,) + spatial_shape)
            v[write_idx, ...] = coarse_da.values.astype(np.float32, copy=False)

            # then resize time, write time (commit)
            tvar._h5ds.resize((write_idx + 1,))
            tvar[write_idx] = s

            saved.add(s)
            appended += 1

    finally:
        if nc_app is not None:
            nc_app.close()

    print(
        f"\nUpdated NetCDF (one var): {out_nc}\n"
        f"  var: {varname}\n"
        f"  appended: {appended}\n"
        f"  skipped (already saved): {skipped_saved}\n"
        f"  skipped (missing hours): {skipped_missing}\n"
        f"  origin_iter={origin_iter}, step={step}, block_size={block_size}\n"
        f"  coarse box: box_j={box_j}, box_i={box_i}",
        flush=True,
    )


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    path_scratch= "/scratch/jczhang/"
    path_MITgcm= "MITgcm_1km/verification/tutorial_reentrant_channel/output"
    grid_dir = path_scratch + path_MITgcm

    ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)

    dims_map = {
        "U": ("k", "j", "i_g"),
        "V": ("k", "j_g", "i"),
        "W": ("k_l", "j", "i"),
        "T": ("k", "j", "i"),
        "S": ("k", "j", "i"),
    }

    nStep1, nStep2, dS = 1200060 + 180 + 1980, 1279800 + 1, 180
    time_index = np.arange(nStep1, nStep2, dS, dtype=int)
    box_j, box_i = 50, 50

    varname = "W"  # change to "U","V","W","S"

    load_T = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var=varname, dims=dims_map[varname])

    out_nc = f"./{varname}_12hmean_coarse.nc"

    _12h_mean_to_netcdf_onevar(
        start_time_index=nStep1,
        end_time_index=nStep2,
        load_snapshot=load_T,
        out_nc=out_nc,
        ds_fine=ds_fine,
        box_j=box_j,
        box_i=box_i,
        varname=varname,
        block_size=6,
        step=dS,
        time_name="time",
        skip_missing=False,
        print_every=1,
    )


[try] n=0/71  block_start=1202220
[try] n=1/71  block_start=1203300
[try] n=2/71  block_start=1204380
[try] n=3/71  block_start=1205460
[try] n=4/71  block_start=1206540
[try] n=5/71  block_start=1207620
[try] n=6/71  block_start=1208700
[try] n=7/71  block_start=1209780
[try] n=8/71  block_start=1210860
[try] n=9/71  block_start=1211940
[try] n=10/71  block_start=1213020
[try] n=11/71  block_start=1214100
[try] n=12/71  block_start=1215180
[try] n=13/71  block_start=1216260
[try] n=14/71  block_start=1217340
[try] n=15/71  block_start=1218420
[try] n=16/71  block_start=1219500
[try] n=17/71  block_start=1220580
[try] n=18/71  block_start=1221660
[try] n=19/71  block_start=1222740
[try] n=20/71  block_start=1223820
[try] n=21/71  block_start=1224900
[try] n=22/71  block_start=1225980
[try] n=23/71  block_start=1227060
[try] n=24/71  block_start=1228140
[try] n=25/71  block_start=1229220
[try] n=26/71  block_start=1230300
[try] n=27/71  block_start=1231380
[try] n=28/71  block_start=123